# 09_binding_pocket_analysis

Notebook UI for UPO homolog pocket analysis using an LLM with binding, alignment, and optional reaction inputs.

## Python Path Setup
Ensure project-root imports work whether Jupyter starts from repo root or `notebooks/`.

In [12]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))


## Imports
Load helper functions for table loading, LLM analysis, output export, and thread persistence.

In [13]:
import importlib
import agentic_protein_design.steps.analyze_binding_pocket as bp
bp = importlib.reload(bp)
from agentic_protein_design.core.thread_context import load_optional_thread_context
from agentic_protein_design.core import apply_notebook_markdown_style, resolve_input_path

analyze_pocket_profiles = bp.analyze_pocket_profiles
default_user_inputs = bp.default_user_inputs
build_prompt_with_context = bp.build_prompt_with_context
generate_llm_pocket_analysis = bp.generate_llm_pocket_analysis
generate_llm_mutation_design_proposal = bp.generate_llm_mutation_design_proposal
run_llm_pocket_analysis_stages = bp.run_llm_pocket_analysis_stages
prompt_3 = bp.prompt_3
init_thread = bp.init_thread
load_input_tables = bp.load_input_tables
persist_thread_update = bp.persist_thread_update
save_llm_analysis = bp.save_llm_analysis
save_mutation_design_proposal = bp.save_mutation_design_proposal
save_rational_engineering_proposals = bp.save_rational_engineering_proposals
save_binding_outputs = bp.save_binding_outputs
setup_data_root = bp.setup_data_root
get_step_processed_dir = bp.get_step_processed_dir
REQUIRED_SUBFOLDERS = bp.REQUIRED_SUBFOLDERS

apply_notebook_markdown_style(font_size_px=14, line_height=1.4)


## User Inputs
Edit all run parameters here (single place): dataset root, thread selection, analysis options, model, and input paths.

In [14]:
root_key = "ECOHARVEST" # "examples"
existing_thread_key = 'binding_pocket_llm_analysis_lipases_b4e2b28e48164985a8e4515ff0b0d417' # "binding_pocket_llm_analysis_UPOs_59353c876ab140688b1c239a15aac24e"  # None

user_inputs = {
    "selected_positions": None, # [100, 103, 104, 107, 141, 222],
    "pairwise_comparisons":  [("RML", "TLL")], # [("CviUPO", "ET096")], # None
    "focus_question": (
        "Identify per-protein structural interpretations and cross-homolog patterns "
        "that could explain activity/property differences."
    ),
    "design_requirements": (
        "Backbone: RML. Goal: improve esterification of oleic acid with sucrose sugar in an environment containing water to enhance transport of the sugar."
        # "Backbone: ET096. Goal: improve peroxygenative mono-oxidation selectivity on S82 while retaining useful activity and limiting over-oxidation to Di-Ox. Prioritize conservative, mechanistically justified mutations and a first-round panel <= 12 variants."
    ),
    "literature_context_thread_key": "literature_review_lipases_b713603189544094bf0c3aea97730dd1", #"literature_review_UPOs_d762a72ec7f04bec9b66ccd3aac21b91",  # Optional: literature-review thread key
    "reaction_data_description": "",
    # "reaction_data_description": (
    #     "- Veratryl alcohol: peroxygenative\n"
    #     "- Naphthalene: peroxygenative\n"
    #     "- NBD: peroxygenative\n"
    #     "- ABTS: peroxidative\n"
    #     "- S82: mixed; Mono-Ox ~ peroxygenation-biased, Di-Ox ~ peroxidation-biased\n"
    #     "Use ratios (e.g. Mono-Ox : Di-Ox) to infer peroxygenation vs peroxidation balance."
    # ),
    "use_reaction_data": False, # True,
    "llm_model": "gpt-5.2",
    "llm_temperature": 0.2,
    "llm_max_rows_per_table": 300,
}

input_paths = {
    # Paths are relative to the data root from project_config.variables.address_dict[root_key].
    "binding_csv": "pdb/lipases/bindingpocket_analysis.csv", # "pdb/bindingpocket_analysis.csv",
    "alignment_csv": "msa/lipases/RML_TLL_ali_withDist_FILT.csv", # "msa/UPO_peroxygenation_ali_withDist_FILT.csv",
    "reaction_data_csv": "expdata/substrate_reaction_data.csv",
}

# Optional: reset analysis options from helper defaults
# user_inputs = default_user_inputs()


## Setup Runtime Context
Initialize data directories and active chat thread from the values above.

In [4]:
data_root, resolved_dirs = setup_data_root(root_key, REQUIRED_SUBFOLDERS)
step_processed_dir = get_step_processed_dir(resolved_dirs)
thread, threads_preview = init_thread(root_key, existing_thread_key)
thread_id = thread["thread_id"]
data_root, step_processed_dir, thread_id


(PosixPath('/Users/charmainechia/Documents/projects/ECOHARVEST'),
 PosixPath('/Users/charmainechia/Documents/projects/ECOHARVEST/processed/09_binding_pocket_analysis'),
 'b4e2b28e48164985a8e4515ff0b0d417')

## Load Input Tables
Load descriptor and alignment tables, and optional reaction data, from `input_paths`.

In [5]:
binding_csv = resolve_input_path(data_root, input_paths["binding_csv"])
alignment_csv = resolve_input_path(data_root, input_paths["alignment_csv"])
reaction_data_csv = None
if user_inputs.get("use_reaction_data", False) and input_paths.get("reaction_data_csv", "").strip():
    reaction_data_csv = resolve_input_path(data_root, input_paths["reaction_data_csv"])

pocket, ali, reaction_df = load_input_tables(binding_csv, alignment_csv, reaction_data_csv)
binding_csv, alignment_csv, reaction_data_csv, pocket.head(3), (None if reaction_df is None else reaction_df.head(3))


(PosixPath('/Users/charmainechia/Documents/projects/ECOHARVEST/pdb/lipases/bindingpocket_analysis.csv'),
 PosixPath('/Users/charmainechia/Documents/projects/ECOHARVEST/msa/lipases/RML_TLL_ali_withDist_FILT.csv'),
 None,
          struct_name      struct_name.1      struct_name.2  \
 0  RML_SucroseOleate  RML_SucroseOleate  RML_SucroseOleate   
 1  TLL_SucroseOleate  TLL_SucroseOleate  TLL_SucroseOleate   
 
    num_pocket_res_ali  num_pocket_res<8  reactive_center_distance  \
 0                  57                51                     6.385   
 1                  60                48                     5.974   
 
    median_dist_res_to_ligand_reactive_center  median_min_dist_res_to_ligand  \
 0                                     10.261                          5.004   
 1                                     10.356                          5.537   
 
    mean_min_dist_to_centroid (distal)  mean_min_dist_to_centroid (proximal)  \
 0                               8.856                 

## Structured Exports
Generate heuristic comparative tables and export CSVs to `processed/`.

In [6]:
selected_positions = user_inputs["selected_positions"]
interp_df, pattern_summary = analyze_pocket_profiles(pocket, ali, selected_positions)
out_interp, out_patterns = save_binding_outputs(interp_df, pattern_summary, step_processed_dir)


## LLM Pocket Analysis
Query the LLM client with the full prompt and input tables, then save markdown output.


In [7]:
# Run two-stage LLM analysis
stage_outputs = run_llm_pocket_analysis_stages(pocket, ali, reaction_df, user_inputs)
prompt_2_output = stage_outputs["prompt_2_output"]
llm_analysis = stage_outputs["combined_analysis"]

out_llm = save_llm_analysis(llm_analysis, step_processed_dir)

# Prompt 3 defaults (overwritten in the next cell)
mutation_design_text = ""
out_mutation_design = None
literature_context_thread_key = None

print(out_llm)


### Binding Pocket Analysis - Stage 1

<details><summary>Prompt</summary>

```text

Analyse the uploaded inputs for a set of proteins to interpret how binding-pocket structure relates to catalytic activity and selectivity. 
Consider how both the proximal (<6 Å from docked ligand) and distal (up to ~11 Å from binding pocket centroid) residues affect the binding pocket environment.

INPUTS
- binding_pocket_table: extracted binding-pocket properties (per protein), calculated separately over proximal and distal residue sets where available.
- pocket_alignment_table: filtered residue alignment of pocket-proximal positions.
- reaction_data (optional): enzyme activity data on substrates.

OBJECTIVE
For each protein, integrate structural descriptors with (optional) reaction data to infer mechanistic behavior and classify pocket phenotypes.

TASKS

1) For each protein:
   - Generate a punchy tagline.
   - Provide a concise 5-6 bullet summary addressing:
        (i) proximal electrostatics  
        (ii) proximal sterics  
        (iii) distal electrostatics  
        (iv) distal sterics / outer pocket size  
        (v) overall synthesis of pocket phenotype, integrating structural properties with catalytic implications:
            - Interpret how geometry and chemistry influence productive (peroxygenative) vs competing (peroxidative) pathways.
            - If reaction_data is provided, use it to support structure–function relationships.

   Use the following column groups:

   PROXIMAL ELECTROSTATICS
   - charged_fraction (proximal), polar_fraction (proximal)
   - kd_weighted (proximal), hw_weighted (proximal)
   - median_dist_res_to_ligand_reactive_center

   PROXIMAL STERICS
   - mean_volume (proximal), weighted_mean_volume (proximal)
   - volume_variance (proximal)
   - small_residue_frac (proximal), bulky_residue_frac (proximal)
   - median_min_dist_res_to_ligand
   - reactive_center_distance
   - num_pocket_res_lt6

   DISTAL ELECTROSTATICS
   - charged_fraction (distal), polar_fraction (distal)
   - kd_weighted (distal), hw_weighted (distal)

   DISTAL STERICS / OUTER POCKET SIZE
   - mean_dist_to_centroid
   - mean_min_dist_to_centroid
   - mean_dist_backbone_to_centroid
   - mean_volume (distal)
   - volume_variance (distal)
   - small_residue_frac (distal), bulky_residue_frac (distal)
   - num_pocket_res_ali

   If proximal/distal suffixes are not explicitly present, infer proximal/distal groupings from context and state your assumption briefly.

2) Comparative analysis requirements (do BOTH):
   A) Intra-protein variant analysis (MANDATORY when variants are present):
   - Detect proteins that share the same base protein identity but differ by variant/mutation labels.
   - For each such protein family, explicitly compare each variant against its WT/reference form (if WT/reference is present).
   - If WT is not explicitly labeled, infer the closest reference sequence in that family and state the assumption.
   - For each variant-vs-reference comparison, report which structural dimensions changed:
        (i) proximal electrostatics
        (ii) proximal sterics
        (iii) distal electrostatics
        (iv) distal sterics / outer pocket size
   - Provide a mechanistic rationale linking those differences to functional shifts.

   B) User-requested pairwise comparisons:
   - Pairwise comparisons requested: RML vs TLL
   - Perform each requested pairwise comparison in addition to section A.
   - Explicitly contrast which structural dimensions changed (prox electrostatics, prox sterics, distal electrostatics, distal sterics).
   - Provide a mechanistic rationale for functional shifts.

3) Distill cross-protein trends or clusters (“pocket phenotypes”):
   - Identify recurring structural archetypes (e.g., tight/polar pose-locking vs open/hydrophobic permissive).
   - Link clusters to turnover vs selectivity trade-offs.

OUTPUT STYLE
- Clear, human-interpretable, mechanistically grounded.
- Emphasize intuition over raw numbers.
- Keep summaries compact and comparative.


REACTION CONTEXT (OPTIONAL)
If reaction_data is provided, use it to support structure–function reasoning.

REACTION_DATA_STATUS: not provided.
```
</details>

#### Response

(Stage 1 output included in compact combined view below.)

### Binding Pocket Analysis - Stage 2

...

#### Response

(Stage 2 output included in compact combined view below.)

### Combined Pocket Analysis

## Stage 1: Global Pocket Phenotypes

Assumptions / parsing notes  
- Your `binding_pocket_table` already contains explicit “(proximal)” and “(distal)” suffixes for most descriptors; I treat those as the <6 Å vs ~6–11 Å residue sets you defined.  
- `num_pocket_res<8` is used as a proxy for “how many residues are close enough to matter”; you requested `num_pocket_res_lt6`, but it isn’t present—so I interpret `num_pocket_res<8` as a *looser* proximity count and lean more on distance metrics (`median_min_dist_res_to_ligand`, `reactive_center_distance`) for true proximity.

---

## Per-protein interpretations

### 1) **RML_SucroseOleate**
**Tagline:** *Polar-rimmed, moderately tight pocket that “guides” the sugar while keeping the acyl chain comfortable.*

- **(i) Proximal electrostatics**
  - Proximal charged/polar fractions are fairly high (charged ~0.18; polar ~0.46), suggesting a chemically “wet” microenvironment near the ligand—helpful for positioning sucrose hydroxyls for deacylation (productive synthesis step).
  - Proximal hydropathy is moderately hydrophobic (hw_weighted ≈ -0.40), consistent with a lipase pocket that still tolerates an oleate chain.
  - Proximal kd_weighted is negative (~ -0.16), i.e., overall more hydrophilic/less hydrophobic character than the distal shell—again consistent with a polar alcohol-acceptor region.
  - Median distance of residues to the ligand reactive center is ~10.26 Å (this is relatively large), implying that *many* pocket residues are not tightly “reactive-center clamping”; catalysis likely depends on a smaller subset of key proximal residues plus dynamics (lid/open state).

- **(ii) Proximal sterics**
  - Mean/weighted proximal residue volume ~103/101 Å³ with moderate variance (~881): not extremely tight, but not highly heterogeneous.
  - Bulky residue fraction proximal ~0.34 (weighted ~0.31) with small-residue fraction ~0.25 → a mixed lining that can provide both shape and some compliance.
  - Median minimum distance residue→ligand ~5.00 Å and reactive_center_distance ~6.39 Å: the docked pose is not “deeply buried” against many sidechains; suggests more of a channel/groove binding mode than a snug cavity lock.
  - `num_pocket_res<8` is high (51), consistent with a fairly extensive pocket surface contacting/near the ligand (even if not all are within 6 Å).

- **(iii) Distal electrostatics**
  - Distal shell is slightly *less* polar/charged than proximal (charged ~0.175; polar ~0.456 ~same), and kd_weighted becomes slightly positive (~0.02), i.e., more hydrophobic character outward.
  - This “polar inside / more hydrophobic outside” gradient is consistent with lipase architecture: polar features to manage the alcohol acceptor chemistry, hydrophobic features to stabilize acyl chain occupancy and lid-open state.

- **(iv) Distal sterics / outer pocket size**
  - Distal centroid distances are modest (mean_dist_to_centroid ~10.69 Å; mean_min_dist_to_centroid ~8.86 Å), indicating an outer pocket that is not extremely expanded.
  - Distal mean volume ~102.5 Å³ with variance ~909: similar to proximal—suggesting the pocket doesn’t flare dramatically outward.

- **(v) Pocket phenotype → catalytic implications (peroxygenative vs peroxidative framing)**
  - Phenotype: **balanced amphiphilic channel**—enough polarity near the ligand to support productive positioning of sucrose OH groups, while maintaining a hydrophobic “runway” for oleate.
  - Mechanistic expectation: this kind of pocket tends to **favor productive binding geometries** (less nonspecific oxidation chemistry) because polar proximal residues can enforce orientation/anchoring of the polyol headgroup. If the pocket were too hydrophobic and open, you’d expect more nonproductive poses and side reactions; RML here looks more “pose-guiding” than “promiscuously permissive.”

---

### 2) **TLL_SucroseOleate**
**Tagline:** *Bulkier, more hydrophobic proximal clamp with a roomier outer shell—built for monoacylation-style positioning rather than deep polar anchoring.*

- **(i) Proximal electrostatics**
  - Proximal polar fraction is lower than RML (polar ~0.41 vs ~0.46), while charged fraction is similar (~0.18).
  - Proximal hw_weighted is more negative (~ -0.50), i.e., **more hydrophobic** near the ligand than RML.
  - Proximal kd_weighted is less negative (~ -0.064 vs -0.158), which partially offsets the hydropathy read; net interpretation: **TLL proximal region is less “polyol-friendly” by polarity but still not extremely hydrophobic by kd**—suggesting fewer strong polar anchoring points but not a purely greasy tunnel.

- **(ii) Proximal sterics**
  - Proximal mean/weighted volume is larger (~108/104 Å³) and variance is higher (~1112): **more sterically structured and heterogeneous**.
  - Bulky residue fraction proximal is notably higher (0.477; weighted 0.414) with slightly fewer small residues (0.227): this reads like a **more shape-defining clamp** near the ligand.
  - Median minimum distance residue→ligand is larger (~5.54 Å vs 5.00 Å in RML) even though reactive_center_distance is slightly shorter (~5.97 Å vs 6.39). This combination often indicates: fewer close sidechain contacts overall, but the reactive center sits somewhat closer to the catalytic machinery while the rest of the ligand is less snugly packed.

- **(iii) Distal electrostatics**
  - Distal charged fraction is higher than RML (0.20 vs 0.175) but distal polar fraction is lower (0.40 vs 0.456).
  - Distal hw_weighted ~ -0.40 (less hydrophobic than its own proximal region), suggesting a **hydrophobic “inner clamp” with a slightly more mixed outer shell**.

- **(iv) Distal sterics / outer pocket size**
  - Distal centroid distances are larger than RML (mean_dist_to_centroid ~11.29 Å vs 10.69; mean_min_dist_to_centroid ~9.38 vs 8.86): **roomier outer pocket / more expanded shell**.
  - Distal volume variance is higher (~1060 vs 909): more geometric diversity outward—often correlated with broader substrate tolerance but also more pose degeneracy.

- **(v) Pocket phenotype → catalytic implications (peroxygenative vs peroxidative framing)**
  - Phenotype: **hydrophobic, bulky proximal “gate” + expanded outer shell**.
  - Mechanistic expectation: this architecture tends to **favor selective, geometry-driven outcomes** when the substrate can be “presented” correctly (e.g., primary-OH targeting on sucrose) because bulky proximal residues can restrict which hydroxyl approaches the acyl-enzyme. At the same time, the lower proximal polarity may reduce strong polyol anchoring, making activity more dependent on lid dynamics and transient binding.
  - In the sugar-ester context (literature you included): TLL is often associated with **6-O monoacylation selectivity** on sucrose; a plausible structural rationale is exactly this: **a shape-selective proximal clamp** that biases which sucrose OH can access the reactive center, while the outer pocket remains permissive enough to accommodate the large sucrose headgroup without forcing deep burial.

---

## 2) Comparative analysis

### A) Intra-protein variant analysis
- No variant families are present in the provided dataset (only **RML_SucroseOleate** and **TLL_SucroseOleate**, no WT/mutant labels). Therefore, no WT-vs-variant comparisons can be performed.

### B) Requested pairwise comparison: **RML vs TLL**

**Proximal electrostatics**
- **RML is more polar proximally** (polar ~0.455 vs 0.409) and less hydrophobic by hw_weighted (≈ -0.404 vs -0.497).
- Mechanistic implication: RML should provide **better polar “landing pads”** for sucrose hydroxyl organization near the catalytic center, potentially improving productive deacylation geometry (synthesis step) and reducing reliance on purely hydrophobic packing.

**Proximal sterics**
- **TLL is bulkier and more heterogeneous proximally** (higher mean/weighted volume and higher variance; bulky fraction 0.477 vs 0.341).
- Mechanistic implication: TLL likely imposes **stronger shape constraints** on how sucrose can sit near the reactive center—consistent with **regioselective monoacylation tendencies** (restricting which OH can approach).

**Distal electrostatics**
- TLL distal shell: **more charged but less polar** (charged 0.20 vs 0.175; polar 0.40 vs 0.456).
- Mechanistic implication: TLL may have more discrete ionic features at the rim/outer shell (possible steering/solvent interactions), but fewer overall polar contacts—potentially promoting a more “interfacial” binding mode rather than deep polyol solvation.

**Distal sterics / outer pocket size**
- **TLL has a larger outer pocket** (greater centroid distances) and higher distal variance.
- Mechanistic implication: TLL can accommodate the bulky sucrose headgroup with less penalty (less need to thread deeply), but that extra space can increase **pose multiplicity**—making proximal steric gating more important for selectivity.

**Pocket-phenotype conclusion (RML vs TLL)**
- **RML:** “pose-guiding amphiphilic channel” → likely more robust productive binding for polar acceptors (sucrose) when access is achieved.  
- **TLL:** “bulky hydrophobic clamp + roomy shell” → likely stronger regioselective presentation (monoacylation bias) but potentially more dependent on dynamics/solvent to achieve productive sugar engagement.

---

## 3) Cross-protein pocket phenotypes (clusters)

With only two proteins, the “clusters” reduce to two archetypes:

1) **Amphiphilic, polarity-supported acceptor binding (RML-like)**
- Hallmarks: higher proximal polar fraction; less hydrophobic proximal hw; moderate steric bulk.
- Trade-off intuition: tends toward **more reliable productive chemistry** for polar acceptors (better orientation/anchoring), potentially at the cost of **less extreme shape-enforced regioselectivity**.

2) **Hydrophobic, bulky proximal gating with expanded outer shell (TLL-like)**
- Hallmarks: higher proximal bulky fraction and variance; more hydrophobic proximal hw; larger distal centroid distances.
- Trade-off intuition: tends toward **higher selectivity via steric presentation** (which OH can reach), but can show **greater sensitivity to lid/open-state population and solvent microenvironment** because polar anchoring is weaker.

If you add more homologs/variants (especially CALB/CALA or RML/TLL mutants), I can turn these into a multi-cluster map (e.g., tight/polar “pose-locking” vs open/hydrophobic “permissive”) and explicitly assign each protein to a phenotype with engineering levers (which residues to polarize, which to debulk, which to widen/narrow distally).

## Stage 2: Residue-Level Mechanistic Drivers

## 1) Key variable pocket positions → residue-level mechanistic hypotheses (RML vs TLL)

Below, “key” means (i) within the filtered pocket set and (ii) plausibly causal for the **RML polar/pose-guiding channel** vs **TLL hydrophobic/bulky proximal clamp + roomier shell** phenotype described in the structural summary.

### A. **RML 83 (S) ↔ TLL 84 (R)**  *(min dist ~3.5–3.6 Å; proximal)*
- **Residues:** RML **Ser83** vs TLL **Arg84**
- **Substitution class:** **Electrostatic + steric** (neutral small polar → **positively charged, bulky**)
- **Mechanistic consequence:**
  - In **TLL**, Arg introduces a **localized cationic “hook”** and a larger sidechain that can **sterically gate** nearby sugar hydroxyls. This matches the summary’s **shape-defining proximal clamp** and can bias which sucrose OH can approach the acyl-enzyme (consistent with TLL’s monoacylation/regioselectivity tendency).
  - In **RML**, Ser keeps this region **smaller and more H-bond permissive without strong ionic steering**, consistent with a **more uniformly polar rim** that “guides” sucrose rather than clamping it.
- **Phenotype tie-back:** This single change can simultaneously explain **(i) higher proximal steric bulk/heterogeneity in TLL** and **(ii) reduced “polyol-friendly” polarity (fewer neutral H-bond donors/acceptors arranged as a network) despite similar charged fraction overall**.

---

### B. **RML 91 (D) ↔ TLL 92 (N)**  *(min dist ~2.37 vs 3.30 Å; very proximal)*
- **Residues:** RML **Asp91** vs TLL **Asn92**
- **Substitution class:** **Electrostatic** (negative → neutral polar amide) + **polarity shift** (ionic → H-bonding)
- **Mechanistic consequence:**
  - **RML Asp91** provides a **fixed negative charge** very close to the ligand. That can create a **strong electrostatic anchor/steering point** for sucrose OH patterning (via direct H-bonds or water-mediated networks), supporting the summary’s **polar proximal “landing pads”** and more reliable productive binding geometries.
  - **TLL Asn92** removes the negative charge, weakening ionic steering and making binding more dependent on **steric presentation** (the “clamp”) and dynamics. This aligns with the summary’s view that TLL has **weaker polar anchoring** near the ligand.
- **Mechanistic prediction:** D→N in this location should **increase pose degeneracy** and potentially **increase reliance on proximal bulky residues to enforce regioselectivity** (i.e., selectivity maintained by shape rather than electrostatic anchoring).

---

### C. **RML 215 (F) ↔ TLL 213 (Y)**  *(min dist ~6.8 Å; distal/moderately close)*
- **Residues:** RML **Phe215** vs TLL **Tyr213**
- **Substitution class:** **Polarity shift** (hydrophobic aromatic → aromatic with **phenolic OH**)
- **Mechanistic consequence:**
  - **TLL Tyr213** can add a **rim H-bond donor/acceptor** that may interact with sucrose at the **outer shell**, consistent with the summary’s **more charged/mixed distal shell** and “interfacial” binding mode.
  - **RML Phe215** keeps this region more purely hydrophobic/aromatic, consistent with RML’s **hydrophobic runway outward** while keeping key polarity more proximal.
- **Mechanistic prediction:** This is more likely a **secondary modulator**: it can tune **entry/exit and outer-shell residence time** (and thus effective on-rate/pose filtering), rather than directly controlling reactive-center geometry.

---

### D. **RML 174 (Q) ↔ TLL 171 (Y)**  *(~7.5–7.8 Å; distal)*
- **Residues:** RML **Gln174** vs TLL **Tyr171**
- **Substitution class:** **Steric + polarity shift** (flexible polar amide → bulkier aromatic with phenolic OH)
- **Mechanistic consequence:**
  - **TLL Tyr171** can contribute to the **bulkier, more structured outer shell** (summary: larger distal centroid distances + higher variance). Aromatic packing can create **shape features** that help “stage” the sucrose headgroup without deep burial.
  - **RML Gln174** is more flexible and polar, consistent with a **more continuously polar surface** that can accommodate multiple H-bonding patterns (pose-guiding rather than clamping).
- **Mechanistic prediction:** Likely affects **outer-shell shaping and solvent exposure** of the sugar, influencing **pose multiplicity** and possibly product distribution (mono vs further acylation) indirectly.

---

### E. **RML 265 (T) ↔ TLL 265 (I)**  *(min dist ~2.91 vs 3.97 Å; proximal)*
- **Residues:** RML **Thr265** vs TLL **Ile265**
- **Substitution class:** **Polarity + steric** (small polar → hydrophobic, slightly bulkier)
- **Mechanistic consequence:**
  - **TLL Ile265** increases **local hydrophobicity** near the ligand and removes an H-bonding handle, consistent with the summary’s **more hydrophobic proximal clamp**.
  - **RML Thr265** supports the **polar proximal microenvironment** and could help stabilize a productive sucrose OH orientation (directly or via structured water).
- **Mechanistic prediction:** This position is a plausible contributor to the **RML “polyol-friendly” proximal region** vs **TLL hydrophobic gating**.

---

### F. **RML 264 (N) ↔ TLL 264 (L)**  *(~6–7.9 Å; distal/edge)*
- **Residues:** RML **Asn264** vs TLL **Leu264**
- **Substitution class:** **Polarity shift** (polar → hydrophobic)
- **Mechanistic consequence:**
  - **TLL Leu264** reinforces a **hydrophobic wall** at/near the pocket periphery, consistent with the summary’s **hydrophobic inner clamp** architecture.
  - **RML Asn264** maintains a polar feature that can support **sucrose approach/solvation** at the rim.
- **Mechanistic prediction:** More of a **secondary modulator** (rim wetting/entry energetics) than a direct reactive-center clamp.

---

### G. **RML 303? (L267) ↔ TLL 267 (T)**  *(min dist ~3.78 vs 5.15 Å; proximal-to-mid)*
- **Residues:** RML **Leu267** vs TLL **Thr267**
- **Substitution class:** **Polarity shift** (hydrophobic → polar)
- **Mechanistic consequence (context-dependent):**
  - This is one of the few changes that would make **TLL locally more polar** than RML. If oriented toward the ligand, **Thr267** could partially compensate for TLL’s reduced proximal polarity elsewhere (e.g., D→N at 92; T→I at 265).
  - However, the larger TLL “clamp” phenotype suggests that even if Thr is present, the **net proximal environment** is still more shape/sterics-driven.
- **Mechanistic prediction:** likely a **fine-tuner** of local H-bonding rather than a primary driver.

---

### H. Other variable positions that are mostly steric/background in this context
- **RML 90 (A) ↔ TLL 91 (G):** small↔small; minor packing/dynamics effect.
- **RML 93 (T) ↔ TLL 94 (N):** polar↔polar; modest H-bond pattern change, likely secondary.
- **RML 207 (H) ↔ TLL 205 (R):** charge-capable↔positive; but distances here are ~7 Å min—more likely distal electrostatic steering than direct clamp.
- **RML 254 (V) ↔ TLL 255 (I):** hydrophobic↔hydrophobic; small steric tweak.
- **RML 259 (S) ↔ TLL 260 (W):** big steric change but at ~7.6–7.7 Å; could shape outer shell, but less directly tied to the “proximal clamp” unless this residue points inward in the open state.

---

## 2) Variant-within-family contrasts
Only **two base sequences (RML vs TLL)** are present; there are **no intra-family variants** (WT vs mutants) in the provided alignment table, so I can’t do “point mutations in variants of the same base sequence” comparisons from this dataset.  
If you provide RML-mutant/TLL-mutant rows, the same framework above will map each mutation onto the **polar-channel vs hydrophobic-clamp** axes.

---

## 3) Ranked residue list (mechanistic drivers vs modulators vs likely neutral)

### High-confidence mechanistic driver residues (most likely causal for phenotype differences)
1. **RML Ser83 ↔ TLL Arg84** — introduces **bulky positive gate** in TLL (steric + electrostatic); matches “proximal clamp”.
2. **RML Asp91 ↔ TLL Asn92** — removes **proximal negative anchor** in TLL; matches reduced polyol-friendly anchoring.
3. **RML Thr265 ↔ TLL Ile265** — polar→hydrophobic near ligand; supports RML polar proximal vs TLL hydrophobic clamp.

### Secondary modulators (tune rim wetting, outer-shell shaping, pose multiplicity)
- **RML Asn264 ↔ TLL Leu264** — polar→hydrophobic at rim/edge.
- **RML Gln174 ↔ TLL Tyr171** — flexible polar→aromatic polar; outer-shell shaping.
- **RML Phe215 ↔ TLL Tyr213** — adds phenolic OH; distal H-bonding/entry effects.
- **RML Leu267 ↔ TLL Thr267** — hydrophobic→polar; local compensation/fine-tuning.
- **RML His207 ↔ TLL Arg205** — distal electrostatic steering (context-dependent).

### Likely neutral/background (small effects or conservative swaps in this pocket context)
- **RML Ala90 ↔ TLL Gly91**
- **RML Thr93 ↔ TLL Asn94**
- **RML Val254 ↔ TLL Ile255**
- (Most other listed positions are conserved between RML and TLL in this filtered pocket set.)

If you share the **3D orientation** (sidechain vectors) for the top candidates (83/91/265/264) in the open-state structures, I can tighten these into testable hypotheses (e.g., predicted H-bond partners on sucrose; expected shifts in reactive-center approach angles; which OH becomes sterically excluded).

/Users/charmainechia/Documents/projects/ECOHARVEST/processed/09_binding_pocket_analysis/binding_pocket_llm_analysis.md


## LLM Backbone Engineering Proposal
Use Stage-2 residue-level drivers plus optional literature-thread context to propose mutation designs under user requirements.

In [15]:
design_requirements = str(user_inputs.get("design_requirements", "")).strip()
literature_context_thread_key = str(user_inputs.get("literature_context_thread_key", "")).strip() or None

context_result = load_optional_thread_context(
    literature_context_thread_key,
    include_referenced_files=False,
    max_chars_per_file=40000,
    on_missing="warn",
    json_artifact_names=["engineering_strategy"],
)
literature_context_bundle = context_result.get("context_bundle")
engineering_strategy = ((literature_context_bundle or {}).get("referenced_json_objects") or {}).get("engineering_strategy", {})

mutation_design_outputs = generate_llm_mutation_design_proposal(
    prompt_2_output=prompt_2_output,
    design_requirements=design_requirements,
    user_inputs=user_inputs,
    engineering_strategy=engineering_strategy,
)
mutation_design_text = str(mutation_design_outputs.get("prompt_3_output_text", ""))
proposals_df = mutation_design_outputs.get("proposals_df")
out_mutation_design = save_mutation_design_proposal(mutation_design_text, step_processed_dir)
out_rational_proposals = save_rational_engineering_proposals(proposals_df, step_processed_dir)

{"mutation_design_path": str(out_mutation_design), "rational_engineering_proposals_path": str(out_rational_proposals), "n_rows": 0 if proposals_df is None else int(len(proposals_df))}, proposals_df.head(20) if proposals_df is not None else proposals_df


### Binding Pocket Mutation Design Proposal

<details><summary>Prompt</summary>

```text

You are designing enzyme variants for rational engineering.

You are given:
1) prompt_2_output: residue-level mechanistic analysis of binding-pocket drivers.
2) engineering_strategy (optional): structured literature-derived strategy JSON.
3) design_requirements: user-provided requirements including:
   - target backbone protein to engineer
   - engineering aims (activity/selectivity/stability/pathway bias)
   - constraints (allowed positions, mutation budget, excluded residues/motifs, expression or assay limits)

TASK
Generate a concrete mutation design proposal grounded primarily in prompt_2_output and supported by engineering_strategy when relevant.

OUTPUT FORMAT
1) Design Intent
   - State backbone protein and explicit engineering objective.

2) Proposed Mutations (ranked)
   - Provide 5-10 proposals total.
   - Include both:
     - specific substitutions (e.g., F88L), and
     - optional position-level exploration suggestions (e.g., site-saturation at position 158 with a small focused set).
   - For each proposal provide:
     - rank
     - proposal (mutation or position-set)
     - rationale linked to prompt_2 mechanistic driver(s)
     - which engineering hypothesis from INPUT_DATA_JSON (added below under this prompt) this targets, if relevant
     - expected directional effect on function
     - risk/tradeoff
     - confidence (high/medium/low)

3) Minimal Experimental Plan
   - Suggest a compact first-round panel (6-12 variants max), prioritizing high-information designs.
   - Include a short assay/readout plan aligned with the objective.

4) Rejected Alternatives
   - Briefly list 3-5 plausible but lower-priority options and why they were deprioritized.

RULES
- Do not invent residue numbering outside the provided context.
- Keep causal links explicit from residue-level mechanism -> mutation -> expected phenotype.
- If engineering_strategy conflicts with prompt_2_output, state the conflict and choose a conservative design.
- Prefer practical, testable proposals over speculative broad recommendations.

```
</details>

#### Response

(Full mutation proposal shown below in compact view.)

### Mutation Design Proposal

## 1) Design Intent
- **Backbone protein:** RML (Rhizomucor miehei lipase)
- **Objective:** Increase **esterification of oleic acid with sucrose** under **water-containing conditions** by strengthening **productive sucrose binding/pose guidance** (reduce pose degeneracy, improve near-attack geometry) while maintaining enough pocket accessibility for the bulky sugar.

Grounding from `prompt_2_output`: RML’s advantage is a **polar/pose-guiding proximal channel** (not a hydrophobic clamp). So the most conservative path is to **reinforce/extend proximal polar anchoring** rather than “TLL-ifying” the pocket with hydrophobic gating that could exclude sucrose in water.

---

## 2) Proposed Mutations (ranked)

### 1) **D91E**
- **Rationale (mechanistic driver):** Position **Asp91** is identified as a *very proximal negative anchor* that can electrostatically steer sucrose OH patterning. Extending Asp→Glu can **project the negative charge slightly farther** into the binding region, potentially improving **capture/retention of sucrose in water** and stabilizing a productive pose.
- **Targets hypothesis:** “RML polar proximal landing pad drives productive sucrose binding.”
- **Expected effect:** ↑ sucrose binding/pose stability → ↑ esterification rate/yield in wet media.
- **Risk/tradeoff:** Could over-stabilize nonproductive H-bond networks or perturb local geometry if space is tight.
- **Confidence:** **Medium** (same charge, modest geometric change; effect depends on sidechain orientation).

### 2) **S83T**
- **Rationale (mechanistic driver):** **Ser83** is a key proximal position where TLL has a bulky charged Arg “gate.” For sucrose-in-water, we likely want **more H-bonding without steric exclusion**. Ser→Thr adds a methyl (slightly more shape) while **retaining an OH** to strengthen local H-bonding and subtly bias pose without clamping.
- **Targets hypothesis:** “Proximal polar rim guides sucrose rather than sterically gating it.”
- **Expected effect:** ↑ productive pose frequency; potentially improved regio-bias consistency without losing activity.
- **Risk/tradeoff:** Small steric increase could reduce accessibility if this sidechain points inward.
- **Confidence:** **Medium–High** (conservative, aligned with RML polar-channel concept).

### 3) **T265S**
- **Rationale (mechanistic driver):** **Thr265** is proximal and contributes to RML’s polar microenvironment (TLL has Ile here, more hydrophobic). Thr→Ser keeps polarity but **reduces steric bulk**, potentially allowing sucrose to sit closer while maintaining an H-bond handle (or structured water) near the reactive center.
- **Targets hypothesis:** “Proximal polarity + reduced steric hindrance improves sucrose approach in water.”
- **Expected effect:** ↑ sucrose accommodation/near-attack geometry → ↑ esterification.
- **Risk/tradeoff:** If Thr’s methyl is important for packing, Ser could increase flexibility/pose degeneracy.
- **Confidence:** **Medium**

### 4) **N264Q**
- **Rationale (secondary modulator):** **Asn264** is a rim/edge polar feature (TLL has Leu, more hydrophobic). Asn→Gln can **extend the polar sidechain** to improve “rim wetting” and initial sucrose capture/retention in aqueous environments, potentially improving effective on-rate and residence time.
- **Targets hypothesis:** “Outer-rim polarity supports sucrose entry/retention under water.”
- **Expected effect:** ↑ apparent activity in wet media (better substrate delivery/positioning).
- **Risk/tradeoff:** Added flexibility could increase nonproductive binding; may slightly slow product release.
- **Confidence:** **Low–Medium** (depends strongly on whether 264 points toward solvent/ligand).

### 5) **F215Y**
- **Rationale (secondary modulator):** **Phe215** (RML) vs **Tyr** (TLL) is a distal/moderately close rim position; Tyr adds a phenolic OH that can provide **outer-shell H-bonding** to sucrose, potentially improving staging/entry without changing proximal clamp architecture.
- **Targets hypothesis:** “Distal shell H-bonding increases sucrose residence time and productive entry.”
- **Expected effect:** ↑ binding/retention → modest ↑ esterification in water.
- **Risk/tradeoff:** Could increase water retention locally or alter dynamics; effect likely modest.
- **Confidence:** **Medium**

### 6) **Q174Y**
- **Rationale (secondary modulator):** **Gln174** is distal; TLL has Tyr at the analogous site, contributing to a more structured outer shell. Introducing Tyr could create a **more defined staging surface** for sucrose (aromatic + OH), potentially reducing pose multiplicity before the sugar reaches the proximal polar region.
- **Targets hypothesis:** “Outer-shell shaping can pre-organize sucrose for productive approach.”
- **Expected effect:** Potential ↑ selectivity/pose filtering; may improve mono-ester formation efficiency.
- **Risk/tradeoff:** Added bulk could impede entry for bulky sucrose; could reduce overall turnover if it becomes a bottleneck.
- **Confidence:** **Low–Medium**

### 7) **Focused exploration at 83: {S83T, S83N, S83Q}**
- **Rationale:** 83 is a top mechanistic driver position. Rather than jumping to Arg-like gating (likely harmful for sucrose-in-water), explore **polar, non-cationic** options that can tune H-bond geometry and mild sterics.
- **Targets hypothesis:** “Fine-tune proximal pose guidance without clamp-like exclusion.”
- **Expected effect:** Identify best balance of binding vs accessibility.
- **Risk/tradeoff:** Some variants may reduce activity if they disrupt local packing.
- **Confidence:** **Medium** (position is high-impact; best residue is uncertain).

### 8) **Focused exploration at 265: {T265S, T265N}**
- **Rationale:** 265 is a proximal driver. Keep it polar (avoid Ile-like hydrophobization) but test **smaller (Ser)** vs **amide (Asn)** to modulate H-bonding patterning near sucrose OH.
- **Targets hypothesis:** “Optimize proximal polar microenvironment for sucrose OH steering.”
- **Expected effect:** Potential ↑ catalytic efficiency in wet media.
- **Risk/tradeoff:** Asn could introduce alternative H-bond networks that trap nonproductive poses.
- **Confidence:** **Low–Medium**

---

## 3) Minimal Experimental Plan

### First-round variant panel (10 total; high-information, low combinatorial explosion)
1. **WT RML**
2. **D91E**
3. **S83T**
4. **T265S**
5. **N264Q**
6. **F215Y**
7. **Q174Y**
8. **S83T + D91E** (tests additive strengthening of proximal polar guidance)
9. **S83T + T265S** (tests cooperative proximal tuning without changing charge)
10. **D91E + T265S** (tests “anchor + proximal geometry” synergy)

*(I’m intentionally not proposing D91N, S83R, or T265I in round 1 because prompt_2_output links those directions to the more hydrophobic/steric TLL clamp phenotype, which is risky for sucrose handling in water.)*

### Assay/readouts aligned to goal (water-containing system)
- **Primary activity readout:** Rate and/or yield of **sucrose oleate formation** (e.g., HPLC/UPLC-CAD or LC-MS quantitation of mono-/di-/poly-esters).
- **Selectivity readout:** Product distribution (monoester vs higher esters), since improved productive binding in water often shifts distribution.
- **Water tolerance metric:** Run a small water gradient (e.g., low vs higher water activity) and track **relative activity retention**.
- **Optional mechanistic proxy:** If feasible, measure apparent **Km-like behavior for sucrose** (or initial-rate vs sucrose concentration) to see whether variants improve effective binding/pose formation in wet media.

---

## 4) Rejected Alternatives (lower priority)  

1) **S83R (RML→TLL-like gate)**
- **Why deprioritized:** prompt_2_output ties Arg here to a **bulky cationic hook/steric gate** consistent with a clamp. In water with bulky sucrose, this risks **excluding productive poses** or over-biasing orientation.

2) **D91N (remove proximal negative anchor)**
- **Why deprioritized:** prompt_2_output predicts D→N increases **pose degeneracy** and reduces ionic steering—opposite of what you want for sucrose capture/productive binding in water.

3) **T265I (hydrophobize proximal region)**
- **Why deprioritized:** Mechanistically linked to TLL’s **hydrophobic clamp**; likely reduces “polyol-friendly” character and could worsen performance in aqueous environments.

4) **N264L (rim polar→hydrophobic)**
- **Why deprioritized:** Would reduce rim wetting/entry energetics for sucrose in water; likely counterproductive for transport/capture.

5) **Large distal steric changes like S259W (noted as big but distal)**
- **Why deprioritized:** prompt_2_output flags it as ~7.6–7.7 Å and orientation-dependent; high risk of unintended access/dynamics effects without clear causal linkage to the proximal pose-guiding mechanism.

If you can share which sucrose hydroxyl is the desired acylation site (or your current product distribution), I can bias the proposals toward mutations expected to favor that regio-orientation while keeping the “polar-channel in water” design principle.

({'mutation_design_path': '/Users/charmainechia/Documents/projects/ECOHARVEST/processed/09_binding_pocket_analysis/binding_pocket_mutation_design.md',
  'rational_engineering_proposals_path': '/Users/charmainechia/Documents/projects/ECOHARVEST/processed/09_binding_pocket_analysis/rational_engineering_proposals.csv',
  'n_rows': 16},
                                         mutant  \
 0                                         D91E   
 1                                         S83T   
 2                                        T265S   
 3                                        N264Q   
 4                                        F215Y   
 5                                        Q174Y   
 6   Library at position 83: {S83T, S83N, S83Q}   
 7      Library at position 265: {T265S, T265N}   
 8                                  S83T + D91E   
 9                                 S83T + T265S   
 10                                D91E + T265S   
 11                                        S83R   
 1

 ## Save Thread Update
Run this final cell to append run metadata and prompt context to `chats/<llm_process_tag>_<thread_id>.json`.

In [16]:
persist_thread_update(
    root_key=root_key,
    thread_id=thread_id,
    user_inputs=user_inputs,
    input_paths=input_paths,
    selected_positions=selected_positions,
    reaction_df=reaction_df,
    out_interp=out_interp,
    out_patterns=out_patterns,
    llm_analysis_path=out_llm,
    llm_analysis_text=llm_analysis,
    mutation_design_path=out_mutation_design,
    mutation_design_text=mutation_design_text,
    rational_engineering_proposals_path=out_rational_proposals,
    rational_engineering_proposals_rows=0 if proposals_df is None else int(len(proposals_df)),
    literature_context_thread_key=literature_context_thread_key,
    llm_model=str(user_inputs.get("llm_model", "")),
)


'2026-04-03T16:32:19.012873+00:00'